In [1]:
import pandas as pd
import os
import re
import numpy as np

In [3]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [8]:
file_path = "P:/Dataset/MO-DBT-data-curation/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [9]:
files

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes',
 'procedures']

In [ ]:
shared_path = "P:/Dataset/MO-DBT-data-curation/R3Data"

### 10. procedures
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [11]:
idx = 10
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

procedures
['PATIENT_STUDY_ID', 'PROCEDURE_DATE', 'PROCEDURE_CODE', 'PROCEDURE_TYPE', 'PROCEDURE_NAME', 'PROCEDURE_LOCATION', 'ORDER_DATE']


### **Cancer**

In [12]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['procedures.txt', 'procedures.csv', 'procedures.csv']

In [13]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

#### <span style="color: cyan;">**Interfile de-duplicate**</span> 

In [14]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [15]:
final_df

,PATIENT_STUDY_ID,PROCEDURE_DATE,PROCEDURE_CODE,PROCEDURE_TYPE,PROCEDURE_NAME,PROCEDURE_LOCATION,ORDER_DATE
0,4335275754,06/08/2020,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,05/28/2020
1,4335887037,01/16/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,01/03/2018
2,4335887037,12/24/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,12/24/2018
3,4335887037,12/24/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,12/24/2018
4,4335887037,01/03/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,01/03/2018
...,...,...,...,...,...,...,...
143153,4334569535,02/20/2020,ECR00028,NaN,NaN,NaN,02/20/2020
143154,4334641922,09/20/2019,ECR00028,NaN,NaN,NaN,10/16/2019
143155,4334000099,02/14/2019,ECR00028,NaN,NaN,NaN,02/14/2019
143156,4333636163,12/28/2022,ECR00028,NaN,NaN,NaN,12/28/2022


In [17]:
final_df['PROCEDURE_DATE'] = pd.to_datetime(final_df['PROCEDURE_DATE'], format='%m/%d/%Y')
final_df['PROCEDURE_DATE'] = final_df['PROCEDURE_DATE'].dt.strftime('%Y-%m-%d')
final_df['ORDER_DATE'] = pd.to_datetime(final_df['ORDER_DATE'], format='%m/%d/%Y')
final_df['ORDER_DATE'] = final_df['ORDER_DATE'].dt.strftime('%Y-%m-%d')

In [18]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'ORDER_DATE', 'PROCEDURE_DATE'], ascending=[True, True, True], inplace=True, ignore_index=True)

#### <span style="color: cyan;">**Remove duplicated entries after interfile deduplicate , add a column marked how many duplicates**</span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [19]:
unique_df = final_df.copy()

In [20]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [21]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [22]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [24]:
unique_df['duplicate_count'].unique()

array([1, 2, 3, 4, 5, 6, 7], dtype=int64)

In [26]:
unique_df

,PATIENT_STUDY_ID,PROCEDURE_DATE,PROCEDURE_CODE,PROCEDURE_TYPE,PROCEDURE_NAME,PROCEDURE_LOCATION,ORDER_DATE,duplicate_count
0,4330018595,2019-08-19,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2019-07-29,1
1,4330018595,2020-02-28,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-02-28,1
2,4330018595,2020-06-02,76642,CPT,"Ultrasound, breast, unilateral, real time with...",NaN,2020-06-02,1
3,4330018595,2020-06-02,77063,CPT,"Screening digital breast tomosynthesis, bilate...",NaN,2020-06-02,1
4,4330018595,2020-06-02,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2020-06-02,1
...,...,...,...,...,...,...,...,...
136805,4339945522,2020-04-13,76642,CPT,"Ultrasound, breast, unilateral, real time with...",NaN,2020-04-13,1
136806,4339945522,2020-04-13,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-04-13,1
136807,4339959661,2019-07-03,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2019-07-03,1
136808,4339959661,2019-09-20,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2019-09-20,1


In [39]:
unique_df.drop(columns="duplicate_count", inplace=True)

In [32]:
unique_df['PROCEDURE_CODE'].nunique(), unique_df['PROCEDURE_NAME'].nunique()

(59, 55)

In [33]:
# Extract unique procedure code and name pairs
procedure_df = (
    unique_df[['PROCEDURE_CODE', 'PROCEDURE_NAME']]
    .drop_duplicates()
    .sort_values('PROCEDURE_CODE')
    .reset_index(drop=True)
)

#### <span style="color: orange;">**SAVE**</span> files

In [37]:
procedure_df.to_excel("P:\Onedrive\R01-MO-DBT\MO-DBT-data-curation\Data\procedures_example.xlsx", index=False)

In [40]:
unique_df.to_excel("P:\Dataset\MO-DBT-data-curation\Cancer\Cleaned\procedures.xlsx", index=False)

### **Control**

In [41]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv"
    ]
file_names

['procedures.txt', 'procedures.csv']

In [42]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

#### <span style="color: cyan;">**Interfile de-duplicate**</span> 

In [43]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

file not exist in: R3_3787_Lee_Control_Extract_Files
file not exist in: R3_3787_Lee_Data_Controls_20240508
✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [44]:
final_df

,PATIENT_STUDY_ID,PROCEDURE_DATE,PROCEDURE_CODE,PROCEDURE_TYPE,PROCEDURE_NAME,PROCEDURE_LOCATION,ORDER_DATE
0,4333652040,11/12/2019,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,11/12/2019
1,4333652040,11/26/2019,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,11/22/2019
2,4334194213,12/20/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,12/18/2018
3,4334194213,08/20/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,08/20/2018
4,4334392590,11/28/2018,19081,CPT,"Biopsy, breast, with placement of breast local...",NaN,11/28/2018
...,...,...,...,...,...,...,...
88658,4334588531,05/31/2017,ECR00028,NaN,NaN,NaN,05/31/2017
88659,4333636163,12/28/2022,ECR00028,NaN,NaN,NaN,12/28/2022
88660,4333396067,06/26/2020,ECR00028,NaN,NaN,NaN,06/26/2020
88661,4334046876,04/24/2017,ECR00028,NaN,NaN,NaN,05/08/2017


In [45]:
final_df['PROCEDURE_DATE'] = pd.to_datetime(final_df['PROCEDURE_DATE'], format='%m/%d/%Y')
final_df['PROCEDURE_DATE'] = final_df['PROCEDURE_DATE'].dt.strftime('%Y-%m-%d')
final_df['ORDER_DATE'] = pd.to_datetime(final_df['ORDER_DATE'], format='%m/%d/%Y')
final_df['ORDER_DATE'] = final_df['ORDER_DATE'].dt.strftime('%Y-%m-%d')

In [46]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'ORDER_DATE', 'PROCEDURE_DATE'], ascending=[True, True, True], inplace=True, ignore_index=True)

#### <span style="color: cyan;">**Remove duplicated entries after interfile deduplicate , add a column marked how many duplicates**</span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [47]:
unique_df = final_df.copy()

In [48]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [49]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [50]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [51]:
unique_df['duplicate_count'].unique()

array([1, 2, 4, 3, 5, 7], dtype=int64)

In [52]:
unique_df

,PATIENT_STUDY_ID,PROCEDURE_DATE,PROCEDURE_CODE,PROCEDURE_TYPE,PROCEDURE_NAME,PROCEDURE_LOCATION,ORDER_DATE,duplicate_count
0,4330018595,2019-08-19,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2019-07-29,1
1,4330018595,2020-02-28,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-02-28,1
2,4330018595,2020-06-02,76642,CPT,"Ultrasound, breast, unilateral, real time with...",NaN,2020-06-02,1
3,4330018595,2020-06-02,77063,CPT,"Screening digital breast tomosynthesis, bilate...",NaN,2020-06-02,1
4,4330018595,2020-06-02,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2020-06-02,1
...,...,...,...,...,...,...,...,...
84500,4339945522,2020-04-13,76642,CPT,"Ultrasound, breast, unilateral, real time with...",NaN,2020-04-13,1
84501,4339945522,2020-04-13,77065,CPT,"Diagnostic mammography, including computer-aid...",NaN,2020-04-13,1
84502,4339959661,2019-07-03,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2019-07-03,1
84503,4339959661,2019-09-20,77067,CPT,"Screening mammography, bilateral (2-view study...",NaN,2019-09-20,1


In [53]:
unique_df.drop(columns="duplicate_count", inplace=True)

In [54]:
unique_df['PROCEDURE_CODE'].nunique(), unique_df['PROCEDURE_NAME'].nunique()

(55, 51)

#### <span style="color: orange;">**SAVE**</span> files

In [55]:
unique_df.to_excel("P:\Dataset\MO-DBT-data-curation\Control\Cleaned\procedures.xlsx", index=False)